# How do you score a match between two lists of numbers?


## The score behind the song

Your music app just played you a song you love, by a band you've never heard of. Somewhere in a data center, two lists of numbers got compared, and the verdict was: same taste.

A song isn't one number. It's loud, fast, sad, bright, a bunch of qualities at once, and your taste is that same tangle. So before we build anything, try the puzzle yourself.

Below are two lists of numbers: your `taste` across four qualities, and a `song_loud` that's simply big across every quality. There's also a `song_match` that's actually close to your taste, quality by quality.

Before you run this: if you score a song by just adding up every number in both lists, which one do you think wins, the song that's actually close to your taste, or the one that's just loud everywhere?


In [ ]:
import numpy as np

features = ["loud", "fast", "sad", "bright"]
taste = np.array([9, 2, 1, 8])
song_match = np.array([8, 3, 2, 7])
song_loud = np.array([9, 9, 9, 9])

naive_score_match = taste.sum() + song_match.sum()
naive_score_loud = taste.sum() + song_loud.sum()

print("naive score, actual match:", round(float(naive_score_match), 2))
print("naive score, just loud:  ", round(float(naive_score_loud), 2))

true_gap_match = np.abs(taste - song_match).sum()
true_gap_loud = np.abs(taste - song_loud).sum()
print("quality-by-quality gap, actual match:", round(float(true_gap_match), 2))
print("quality-by-quality gap, just loud:  ", round(float(true_gap_loud), 2))

assert naive_score_loud > naive_score_match
assert true_gap_match < true_gap_loud

Just adding the lists together gets fooled. The loud song wins on the naive score even though the quality-by-quality gap shows it's actually the worse match. Sheer size can fake a good score. Try answering the puzzle yourself before you read on: what would you do instead? One answer to this runs half of modern machine learning.


## Turn the probe and feel it

Here's the move that lands it. Picture each list as an arrow: each direction is one quality, and the length is how hard the song leans that way. Take one probe arrow (that's you) and four candidate arrows (four songs), placed at these angles from north:

- `probe`: 40 degrees
- `c1`: 75 degrees, length 1.0
- `c2`: 40 degrees, length 1.0
- `c3`: 130 degrees, length 1.0
- `c4`: 220 degrees, length 1.0

The score is: multiply the two arrows entry by entry, then add up the products.

Before you run this: name the one candidate whose score tops out, given those angles.


In [ ]:
def vec(angle_deg, length=1.0):
    a = np.deg2rad(angle_deg)
    return length * np.array([np.cos(a), np.sin(a)])

probe = vec(40)
c1 = vec(75, 1.0)
c2 = vec(40, 1.0)
c3 = vec(130, 1.0)
c4 = vec(220, 1.0)

candidates = {"c1": c1, "c2": c2, "c3": c3, "c4": c4}
scores = {name: float(np.sum(probe * arrow)) for name, arrow in candidates.items()}

for name, score in scores.items():
    print(name, "score:", round(score, 2))

top_candidate = max(scores, key=scores.get)
print("tops out:", top_candidate)

assert top_candidate == "c2"

`c2` shares the probe's exact direction, so it tops out. Multiply entry by entry, add up the products: that's the whole mechanism. This is called the dot product, and NumPy has a shortcut for it, `np.dot`, which we'll check against the manual version shortly.

But watch what happens to a score when nothing about direction changes at all. Before you run this: if you stretch `c1` to 1.3 times its length, without turning it at all, do you think it can overtake `c2`, even though `c2` still points more exactly at the probe?


In [ ]:
c1_before = float(np.sum(probe * c1))
c1_stretched = c1 * 1.3
c1_after = float(np.sum(probe * c1_stretched))
c2_score = float(np.sum(probe * c2))

print("c1 score before stretch:", round(c1_before, 2))
print("c1 score after stretch: ", round(c1_after, 2))
print("c2 score (unchanged):   ", round(c2_score, 2))

assert c1_before < c2_score
assert c1_after > c2_score

`c1` never turned an inch, but stretching it let it overtake `c2`, the candidate that actually points at the probe. Length cheats the meter. The score reads agreement of direction, not sameness, and a long arrow can fake a high number.


## Name the machine you just used

Point the probe along a candidate and the score climbs to its biggest. Point it across, at a right angle, and the score dies to zero. Point it against and the score goes negative.

Before you run this: say what sign you expect (positive, zero, or negative) for a candidate pointing along the probe, one pointing across it, and one pointing directly against it.


In [ ]:
along = vec(40, 1.0)      # same direction as the probe
across = vec(130, 1.0)    # 90 degrees off the probe
against = vec(220, 1.0)   # 180 degrees off the probe

score_along = float(np.sum(probe * along))
score_across = float(np.sum(probe * across))
score_against = float(np.sum(probe * against))

print("along:  ", round(score_along, 2))
print("across: ", round(score_across, 2))
print("against:", round(score_against, 2))

assert score_along > 0
assert np.isclose(score_across, 0.0, atol=1e-9)
assert score_against < 0

Three positions, three readings: agree, ignore, oppose. One multiply and one add, and it behaves like a meter with a needle.

**The dot product is a similarity meter.**

Two lists of numbers go in. One reading of alignment comes out. We write it $a \cdot b$: multiply pairwise, then add. Let's check that our manual version and NumPy's shortcut agree, because that's the whole mechanism, nothing hidden inside it.


In [ ]:
pairs = [(probe, along), (probe, across), (probe, against), (probe, c1_stretched)]

for a, b in pairs:
    manual = np.sum(a * b)
    shortcut = np.dot(a, b)
    assert np.isclose(manual, shortcut)

print("manual multiply-and-add matches np.dot every time.")

## Same meter, different aisle

Rebuild the meter from memory before you read on: two arrows go in, so what two operations turn them into a score?

Now walk that same meter one aisle over. Swap songs for a job posting. Score yourself against what the role wants: multiply the matched entries, add them up. Different aisle, same meter. The multiply-and-add never asked what the numbers meant, and that's exactly why it travels.

Before you run this: given `you = [python_hours, sql_reps, nights_free]` and a `role` that wants similar hours and reps but fewer nights free than you have, do you expect the fit score to land a fairly large positive number, close to zero, or negative?


In [ ]:
job_features = ["python_hours", "sql_reps", "nights_free"]
you = np.array([8, 5, 3])
role = np.array([7, 6, 2])

fit_score = float(np.dot(you, role))
print("fit score:", round(fit_score, 2))

assert fit_score > 0
assert fit_score == float(np.sum(you * role))

Same multiply, same add, new aisle. The recipe doesn't care whether the entries are song qualities or resume hours.

## The honest look

Now the honest look, because a meter you trust blindly is a meter that will lie to you.

Before you run this: if you double one candidate's length exactly, keeping its direction the same, do you think the score doubles exactly, or does it change in some messier way?


In [ ]:
a = probe
b = c1

score_before = float(np.dot(a, b))
b_doubled = b * 2
score_after = float(np.dot(a, b_doubled))

print("score before doubling:", round(score_before, 2))
print("score after doubling: ", round(score_after, 2))

assert np.isclose(score_after, 2 * score_before)

Double one input, double the output, exactly. The meter is linear in each arrow you feed it. That clean scaling is the crack: a long, loud song can win on sheer size, not on agreement, and a padded resume can win on sheer length. Size passes straight through the multiply-and-add, untouched.


## For the walk home

Next time your music app hands you a stranger's song that's exactly right, you'll know the number underneath that said ship it, and you'll know that number can be shouted over.

What is the cheapest fix that keeps the direction and forgets the size?

One thing to try next: take any two of the arrows above and write a version of the score that gives the same reading no matter how long either arrow is. Stretch one of them the way we did with `c1` and check whether your version's reading actually holds still.
